# Project 02 · Neural Decision Boundaries

**Goal.** Build a neural network from NumPy components, then compare how activation and depth change training speed and decision regions.

Every `TODO` is yours. Run each public check immediately after its solution.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

SEED = 7
np.set_printoptions(precision=4, suppress=True)

## 1 · Generate and inspect concentric circles

A line cannot place the inner circle on one side and the outer ring on the other. We generate the data ourselves so the notebook stays self-contained.

In [ ]:
def make_circles(n_samples=1200, noise=0.08, factor=0.42, seed=SEED):
    rng = np.random.default_rng(seed)
    n_inner = n_samples // 2
    n_outer = n_samples - n_inner
    a_outer = rng.uniform(0, 2*np.pi, n_outer)
    a_inner = rng.uniform(0, 2*np.pi, n_inner)
    outer = np.c_[np.cos(a_outer), np.sin(a_outer)]
    inner = factor * np.c_[np.cos(a_inner), np.sin(a_inner)]
    X = np.vstack([outer, inner]) + rng.normal(0, noise, (n_samples, 2))
    y = np.r_[np.zeros(n_outer), np.ones(n_inner)].reshape(-1, 1)
    order = rng.permutation(n_samples)
    return X[order], y[order]

X, y = make_circles()
cut = int(0.8 * len(X))
X_train, y_train = X[:cut], y[:cut]
X_val, y_val = X[cut:], y[cut:]

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(X_train[:, 0], X_train[:, 1], c=y_train[:, 0], cmap='coolwarm', s=16, alpha=.75)
ax.set(title='Training data', xlabel='$x_1$', ylabel='$x_2$', aspect='equal');

**Question 1.** Why can no single straight line solve this dataset? What accuracy would you expect from a model that predicts only one class?

## 2 · One hidden layer, written explicitly

Implement initialization and the cached forward pass

$$Z_1=XW_1+b_1,\quad H_1=\tanh(Z_1),\quad Z_2=H_1W_2+b_2,\quad P=\sigma(Z_2).$$

`predict` should return integer labels using threshold 0.5.

In [ ]:
class OneHiddenNet:
    def __init__(self, input_dim, hidden_dim, seed=SEED):
        # TODO 1: initialize W1, b1, W2, b2 with correct shapes.
        # Use rng = np.random.default_rng(seed) and scale weights by sqrt(1/fan_in).
        raise NotImplementedError

    def forward(self, X):
        # TODO 2: compute z1, h1, z2, probabilities and return (probabilities, cache).
        # Use a numerically stable sigmoid expression by clipping z2 to [-500, 500].
        raise NotImplementedError

    def predict(self, X):
        # TODO 3: threshold the probabilities and return a 1-D integer array.
        raise NotImplementedError

In [ ]:
# Public check 1 · explicit network
net = OneHiddenNet(2, 4, seed=1)
p, cache = net.forward(np.array([[0.2, -0.4], [1.0, 0.5]]))
assert net.W1.shape == (2, 4) and net.b1.shape == (4,)
assert net.W2.shape == (4, 1) and net.b2.shape == (1,)
assert p.shape == (2, 1) and np.all((p > 0) & (p < 1))
assert set(cache) == {'X', 'z1', 'h1', 'z2', 'p'}
assert net.predict(np.zeros((3, 2))).shape == (3,)
print('✓ Check 1 passed')

## 3 · Binary cross-entropy

For targets $y_i\in\{0,1\}$ and predicted probabilities $p_i$,

$$L=-\frac1N\sum_i[y_i\log p_i+(1-y_i)\log(1-p_i)].$$

Clip probabilities inside the function so that neither logarithm receives zero.

In [ ]:
def binary_cross_entropy(y_true, p, eps=1e-12):
    # TODO 4: return the mean binary cross-entropy as a scalar.
    raise NotImplementedError

def binary_cross_entropy_grad(y_true, p, eps=1e-12):
    # This upstream gradient is provided for the training harness.
    p = np.clip(p, eps, 1-eps)
    return (-(y_true / p) + (1-y_true) / (1-p)) / len(y_true)

In [ ]:
# Public check 2 · BCE
yt = np.array([[1.], [0.]])
assert np.isclose(binary_cross_entropy(yt, np.array([[.5], [.5]])), np.log(2))
assert binary_cross_entropy(yt, np.array([[.99], [.01]])) < .02
assert np.isfinite(binary_cross_entropy(yt, np.array([[1.], [0.]])))
print('✓ Check 2 passed')

## 4 · Make the nonlinearity modular

Each activation exposes the same `forward(x)` interface. Implement the three forward transformations. The `backward` methods are provided as infrastructure: **you are not expected to understand or modify them yet**. We will derive them when the course reaches backpropagation.

In [ ]:
class Sigmoid:
    def forward(self, x):
        # TODO 5: cache and return the numerically stable sigmoid output.
        raise NotImplementedError
    def backward(self, grad_out):
        # PROVIDED · Backpropagation infrastructure; study this later.
        return grad_out * self.out * (1 - self.out)

class Tanh:
    def forward(self, x):
        # TODO 6: cache and return tanh(x).
        raise NotImplementedError
    def backward(self, grad_out):
        # PROVIDED · Backpropagation infrastructure; study this later.
        return grad_out * (1 - self.out**2)

class ReLU:
    def forward(self, x):
        # TODO 7: cache x and return ReLU(x).
        raise NotImplementedError
    def backward(self, grad_out):
        # PROVIDED · Backpropagation infrastructure; study this later.
        return grad_out * (self.x > 0)

In [ ]:
# Public checks 3–5 · activation forward outputs
z = np.array([[-2., 0., 2.]])
up = np.ones_like(z)
s = Sigmoid(); so = s.forward(z)
assert np.allclose(so, [[0.11920292, .5, 0.88079708]], atol=1e-7)
t = Tanh(); to = t.forward(z)
assert np.allclose(to, np.tanh(z))
r = ReLU(); assert np.array_equal(r.forward(z), [[0., 0., 2.]])
print('✓ Checks 3–5 passed')

## 5 · A reusable linear layer

Implement only initialization and the familiar affine forward computation $Y=XW+b$. The backward and update methods are supplied so that the experiment can learn. Treat them as a black box for now; later we will derive every line.

In [ ]:
class Linear:
    def __init__(self, in_features, out_features, rng):
        # TODO 8a: initialize W and b; use scale sqrt(1/in_features).
        raise NotImplementedError
    def forward(self, x):
        # TODO 8b: cache x and return the affine output.
        raise NotImplementedError
    def backward(self, grad_out):
        # PROVIDED · Backpropagation infrastructure; study this later.
        self.dW = self.x.T @ grad_out
        self.db = grad_out.sum(axis=0)
        return grad_out @ self.W.T
    def step(self, lr):
        # PROVIDED · Gradient-descent update; study this in optimization.
        self.W -= lr * self.dW
        self.b -= lr * self.db

In [ ]:
# Public check 6 · Linear initialization and forward pass
layer = Linear(2, 3, np.random.default_rng(2))
xx = np.array([[1., 2.], [-1., .5]])
yy = layer.forward(xx)
assert layer.W.shape == (2, 3) and layer.b.shape == (3,)
assert yy.shape == (2, 3)
assert np.allclose(yy, xx @ layer.W + layer.b)
print('✓ Check 6 passed')

## 6 · Compose layers instead of handwriting networks

`Sequential` is provided. Your responsibility is the forward architecture: define a list such as `[Linear, ReLU, Linear, Sigmoid]`. The reverse traversal and learning update are included only so the network can train; we will explain them later.

In [ ]:
class Sequential:
    def __init__(self, layers):
        self.layers = layers
    def forward(self, x):
        for layer in self.layers:
            x = layer.forward(x)
        return x
    def backward(self, grad):
        for layer in reversed(self.layers):
            grad = layer.backward(grad)
        return grad
    def step(self, lr):
        for layer in self.layers:
            if isinstance(layer, Linear):
                layer.step(lr)
    def predict(self, X):
        return (self.forward(X) >= .5).astype(int).ravel()

ACTIVATIONS = {'sigmoid': Sigmoid, 'tanh': Tanh, 'relu': ReLU}

def build_model(depth, activation, width=16, seed=SEED):
    # TODO 9: create exactly `depth` hidden Linear+activation pairs,
    # followed by Linear(width, 1) and Sigmoid(). Return Sequential(layers).
    raise NotImplementedError

In [ ]:
# Public check 7 · architecture declaration
m = build_model(depth=3, activation='tanh', width=8, seed=3)
assert len(m.layers) == 8  # (Linear,Tanh) × 3 + Linear,Sigmoid
assert sum(isinstance(q, Linear) for q in m.layers) == 4
assert m.forward(np.zeros((5, 2))).shape == (5, 1)
print('✓ Check 7 passed')

## 7 · The provided learning and visualization harness

**You do not implement this section.** It takes responsibility for backpropagation and gradient descent so that you can focus on forward computation and model design. We will return to every hidden step in the backpropagation and optimization chapters. Do not change the protocol between models.

In [ ]:
def accuracy(model, X, y):
    return np.mean(model.predict(X) == y.ravel())

def train(model, X, y, epochs=1200, lr=.08):
    losses = []
    start = time.perf_counter()
    for epoch in range(epochs):
        p = model.forward(X)
        loss = binary_cross_entropy(y, p)
        model.backward(binary_cross_entropy_grad(y, p))
        model.step(lr)
        if epoch % 20 == 0 or epoch == epochs-1:
            losses.append((epoch, loss))
    return losses, time.perf_counter() - start

def plot_boundary(ax, model, X, y, title):
    pad = .25
    x0 = np.linspace(X[:,0].min()-pad, X[:,0].max()+pad, 220)
    x1 = np.linspace(X[:,1].min()-pad, X[:,1].max()+pad, 220)
    xx, yy = np.meshgrid(x0, x1)
    grid = np.c_[xx.ravel(), yy.ravel()]
    zz = model.forward(grid).reshape(xx.shape)
    ax.contourf(xx, yy, zz, levels=np.linspace(0,1,21), cmap='coolwarm', alpha=.55)
    ax.contour(xx, yy, zz, levels=[.5], colors='#172431', linewidths=1.5)
    ax.scatter(X[:,0], X[:,1], c=y[:,0], cmap='coolwarm', s=8, edgecolor='none')
    ax.set(title=title, xticks=[], yticks=[], aspect='equal')

## 8 · Run the controlled 3×3 experiment

Rows are activations; columns are hidden-layer depths. Run the cell, then interpret both the table and boundaries. If one configuration fails, report it—do not silently tune only that model.

In [ ]:
depths = [1, 3, 5]
activation_names = ['sigmoid', 'tanh', 'relu']
results, trained = [], {}

for act in activation_names:
    for depth in depths:
        model = build_model(depth, act, width=16, seed=SEED)
        history, seconds = train(model, X_train, y_train, epochs=1200, lr=.08)
        row = {'activation': act, 'depth': depth, 'seconds': seconds,
               'train_acc': accuracy(model, X_train, y_train),
               'val_acc': accuracy(model, X_val, y_val),
               'final_loss': history[-1][1]}
        results.append(row); trained[(act, depth)] = model

header = f"{'activation':>10} {'depth':>5} {'time(s)':>9} {'train':>8} {'valid':>8} {'loss':>10}"
print(header); print('-'*len(header))
for r in results:
    print(f"{r['activation']:>10} {r['depth']:5d} {r['seconds']:9.3f} {r['train_acc']:8.3f} {r['val_acc']:8.3f} {r['final_loss']:10.4f}")

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(13, 12), constrained_layout=True)
for i, act in enumerate(activation_names):
    for j, depth in enumerate(depths):
        r = next(q for q in results if q['activation']==act and q['depth']==depth)
        plot_boundary(axes[i,j], trained[(act,depth)], X_val, y_val,
                      f'{act.title()} · {depth} hidden · val {r["val_acc"]:.3f}')
plt.show()

### Required analysis

1. Which model trained fastest in wall-clock time? Is that comparison alone fair when deeper models perform more matrix multiplications per epoch?
2. Which model achieved the best validation accuracy? Report the training–validation gap.
3. Compare the smoothness and complexity of the nine boundaries. Identify one case where a more complex boundary did **not** provide better validation evidence.
4. Explain one visible difference between sigmoid, tanh, and ReLU using their derivatives.
5. Why can these nine runs not prove that one activation is universally superior? Name at least two variables the protocol held fixed.

> **A flat loss near $\log 2$ is evidence, not permission to hide a run.** If the public gradient checks pass, explain how saturation and the shared learning rate may affect sigmoid. After reporting the controlled comparison, you may run a clearly labeled follow-up that tunes the learning rate separately for each activation.

## 9 · Adventurous extension

Can an algorithm from your previous machine-learning courses produce a better validation result? Try one or more methods such as logistic regression with engineered radial features, $k$-nearest neighbors, an SVM with a nonlinear kernel, or a tree ensemble.

A valid claim of “better” must include: the same train/validation split, the metric, training time, the decision region, and a short explanation of any feature engineering or hyperparameter selection. Do **not** use the test set to choose the method.

## Submission checklist

- [ ] Restart and run all cells from top to bottom.
- [ ] All seven public checks pass.
- [ ] The nine-row results table and 3×3 boundary figure are visible.
- [ ] All five required analysis questions are answered in your own words.
- [ ] The adventurous extension is either completed with fair evidence or explicitly marked as not attempted.